# Anticancer GCN-AISM Pipeline
## Threshold: pIC50 ≥ 7.0 | Dataset: Anticancer_dataset__1_.csv
### All 4 models × 4 conditions (Random/Scaffold × NoAug/Aug) — Full metrics suite


In [1]:
# Install PyTorch Geometric for torch 2.10 + CUDA 12.8

!pip install -q pyg_lib torch_scatter torch_sparse torch_cluster \
torch_spline_conv torch_geometric \
-f https://data.pyg.org/whl/torch-2.10.0+cu128.html

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 40.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 113.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 101.6 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 95.1 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 59.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 6.6 MB/s eta 0:00:00a 0:00:01


In [2]:
# Install RDKit in Kaggle
!pip install -q rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 57.1 MB/s eta 0:00:00:00:0100:01


In [3]:
import os
import glob
import shutil

# Remove old PKL caches
PKL_DIR = "/kaggle/working/checkpoints_anticancer"

if os.path.exists(PKL_DIR):
    for f in glob.glob(f"{PKL_DIR}/*.pkl"):
        os.remove(f)

print("✅ Old PKLs removed")

# Remove old real checkpoints
REAL_CKPT_DIR = "/kaggle/working/real_checkpoints"

if os.path.exists(REAL_CKPT_DIR):
    shutil.rmtree(REAL_CKPT_DIR)

os.makedirs(REAL_CKPT_DIR, exist_ok=True)

print("✅ Real checkpoints cleaned")

✅ Old PKLs removed
✅ Real checkpoints cleaned


In [4]:
from rdkit import Chem
print("RDKit installed successfully")

RDKit installed successfully


In [5]:
# Check torch version
import torch
print(torch.__version__)
print(torch.version.cuda)

2.10.0+cu128
12.8


In [6]:
import glob
import os

for f in glob.glob("/kaggle/working/*.pkl"):
    os.remove(f)

print("✅ Old PKL checkpoints removed")

✅ Old PKL checkpoints removed


In [7]:
import shutil
import os

REAL_CKPT_DIR = "/kaggle/working/real_checkpoints"

if os.path.exists(REAL_CKPT_DIR):
    shutil.rmtree(REAL_CKPT_DIR)

os.makedirs(REAL_CKPT_DIR, exist_ok=True)

print("✅ real_checkpoints cleaned")

✅ real_checkpoints cleaned


In [8]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.data import Data
from torch_geometric.nn import (
    GINEConv, GATv2Conv, TransformerConv, GCNConv,
    global_mean_pool, global_max_pool, GraphNorm, GlobalAttention)
from torch.optim import AdamW
from torch.optim.lr_scheduler import (
    CosineAnnealingWarmRestarts, LinearLR, SequentialLR)
from torch.utils.data import WeightedRandomSampler

import numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, roc_auc_score, matthews_corrcoef, recall_score,
    confusion_matrix, balanced_accuracy_score, f1_score, precision_score,
    r2_score, mean_squared_error, mean_absolute_error,
    roc_curve, precision_recall_curve, average_precision_score)
from sklearn.model_selection import KFold, train_test_split
from sklearn.manifold import TSNE
from scipy import stats
from rdkit import RDLogger, Chem, DataStructs
from rdkit.Chem import AllChem, Descriptors, QED, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold
import warnings, copy, os, pickle

warnings.filterwarnings("ignore")
RDLogger.DisableLog('rdApp.*')
print("✅ Imports OK")


✅ Imports OK


In [9]:
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.set_float32_matmul_precision("high")
torch.backends.cudnn.benchmark = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ── Paths — update DATA_PATH to your CSV location ──────────────────────────
DATA_PATH = "/kaggle/input/datasets/priyanshugoyal2301/anticancer/Anticancer_dataset (1).csv"   # <-- set your path here
CKPT_DIR  = "./checkpoints_anticancer"
REAL_CKPT_DIR = "/kaggle/working/real_checkpoints"
FIG_DIR   = "./figures_anticancer"
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(FIG_DIR,  exist_ok=True)

# ── pIC50 activity threshold (0.7 scale = 7.0 absolute) ────────────────────
ACTIVITY_THRESHOLD = 7.0   # compounds with pIC50 >= 7.0 are labelled ACTIVE

# ── Palettes ────────────────────────────────────────────────────────────────
MODEL_NAMES     = ["GCN", "GAT", "GT", "MSMP"]
MODEL_COLORS    = ["#E74C3C", "#3498DB", "#2ECC71", "#9B59B6"]
COND_COLORS     = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]
COND_LABELS     = ["Random\nNoAug", "Random\nAug", "Scaffold\nNoAug", "Scaffold\nAug"]
COND_NAMES_LONG = ["Random | No Aug", "Random | Aug", "Scaffold | No Aug", "Scaffold | Aug"]
BRIGHT_SEEDS    = ["#FF4757", "#2ED573", "#1E90FF", "#FFA502", "#FF6B81"]

sns.set_theme(style="whitegrid", font_scale=1.15)
plt.rcParams.update({"figure.dpi": 150, "savefig.dpi": 150,
                     "font.family": "DejaVu Sans"})

def savefig(name):
    path = os.path.join(FIG_DIR, name)
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  [saved] {path}")

print(f"✅ Config ready  |  threshold={ACTIVITY_THRESHOLD}  |  CKPT={CKPT_DIR}")


Device: cuda
✅ Config ready  |  threshold=7.0  |  CKPT=./checkpoints_anticancer


In [10]:
def enhanced_atom_features(atom):
    atom_types = ['C','N','O','S','F','Cl','Br','I','P','B']
    f = [1. if atom.GetSymbol()==t else 0. for t in atom_types]
    f += [atom.GetAtomicNum()/100., atom.GetDegree()/8.,
          atom.GetFormalCharge()/5., atom.GetTotalNumHs()/8.,
          atom.GetTotalValence()/8., float(atom.GetIsAromatic()),
          float(atom.IsInRing()),
          float(atom.GetChiralTag()!=Chem.rdchem.ChiralType.CHI_UNSPECIFIED)]
    hyb = atom.GetHybridization()
    for ht in [Chem.rdchem.HybridizationType.SP,
               Chem.rdchem.HybridizationType.SP2,
               Chem.rdchem.HybridizationType.SP3]:
        f.append(1. if hyb==ht else 0.)
    return f   # 21-dim

def enhanced_bond_features(bond):
    bt = bond.GetBondType()
    return [float(bt==Chem.rdchem.BondType.SINGLE),
            float(bt==Chem.rdchem.BondType.DOUBLE),
            float(bt==Chem.rdchem.BondType.TRIPLE),
            float(bt==Chem.rdchem.BondType.AROMATIC),
            float(bond.GetIsConjugated()),
            float(bond.IsInRing()),
            float(bond.GetStereo()!=Chem.rdchem.BondStereo.STEREONONE)]  # 7-dim

def smiles_to_graph(smiles, label, pic50):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    x  = [enhanced_atom_features(a) for a in mol.GetAtoms()]
    ei, ea = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bf = enhanced_bond_features(bond)
        ei += [[i,j],[j,i]]; ea += [bf, bf]
    if not ei:
        ei = [[0,0],[0,0]]; ea = [[0.]*7, [0.]*7]
    try:
        desc = [
            Descriptors.MolWt(mol)/1000.,       Descriptors.MolLogP(mol)/10.,
            Descriptors.TPSA(mol)/200.,          Descriptors.NumRotatableBonds(mol)/20.,
            QED.qed(mol),                        Descriptors.NumHDonors(mol)/10.,
            Descriptors.NumHAcceptors(mol)/10.,  float(rdMolDescriptors.CalcNumAromaticRings(mol))/5.,
            Descriptors.FractionCSP3(mol),       float(mol.GetNumHeavyAtoms())/50.,
            float(rdMolDescriptors.CalcNumRings(mol))/10.,
            min(Descriptors.BertzCT(mol)/1000., 3.),
            float(rdMolDescriptors.CalcNumHeteroatoms(mol))/20.]
    except:
        desc = [0.]*13
    return Data(
        x          = torch.tensor(x,  dtype=torch.float),
        edge_index = torch.tensor(ei, dtype=torch.long).t().contiguous(),
        edge_attr  = torch.tensor(ea, dtype=torch.float),
        desc       = torch.tensor(desc, dtype=torch.float).view(1,-1),
        y_cls      = torch.tensor([label], dtype=torch.float),
        y_reg      = torch.tensor([pic50], dtype=torch.float),
        smiles     = smiles)

print("✅ Feature functions defined  (node_dim=21, edge_dim=7, desc_dim=13)")


✅ Feature functions defined  (node_dim=21, edge_dim=7, desc_dim=13)


In [11]:
df = pd.read_csv(DATA_PATH)

# Adapt column names: dataset uses 'Smiles' and 'PIC50'
df = df.rename(columns={"Smiles": "SMILES", "PIC50": "pIC50"})

# Binary label using threshold 7.0 (pIC50 >= 7.0 → active=1)
df["active"] = (df["pIC50"] >= ACTIVITY_THRESHOLD).astype(int)
df = df.dropna(subset=["pIC50"]).drop_duplicates(subset="SMILES").reset_index(drop=True)

graphs, valid_smiles = [], []
for _, row in df.iterrows():
    g = smiles_to_graph(row.SMILES, row.active, row.pIC50)
    if g is not None:
        graphs.append(g); valid_smiles.append(row.SMILES)

df_filtered = df[df["SMILES"].isin(valid_smiles)].reset_index(drop=True)
y_all = np.array([g.y_reg.item() for g in graphs])
y_reg_mean = float(y_all.mean())
y_reg_std  = float(y_all.std() + 1e-8)
for g in graphs:
    g.y_reg_norm = (g.y_reg - y_reg_mean) / y_reg_std

NODE_DIM = graphs[0].x.shape[1]
DESC_DIM = graphs[0].desc.shape[1]
cc = df_filtered["active"].value_counts()
n_pos = cc.get(1, 1); n_neg = cc.get(0, 1)
pos_weight_val = len(df_filtered) / (2. * n_pos)
neg_weight_val = len(df_filtered) / (2. * n_neg)

print(f"Total graphs  : {len(graphs)}  (node_dim={NODE_DIM}, desc_dim={DESC_DIM})")
print(f"Classes       : {cc.to_dict()}   (threshold pIC50 >= {ACTIVITY_THRESHOLD})")
print(f"pIC50         : mean={y_reg_mean:.3f}   std={y_reg_std:.3f}")
print(f"Class balance : active={n_pos} ({n_pos/len(df_filtered)*100:.1f}%)  "
      f"inactive={n_neg} ({n_neg/len(df_filtered)*100:.1f}%)")


Total graphs  : 3600  (node_dim=21, desc_dim=13)
Classes       : {0: 1800, 1: 1800}   (threshold pIC50 >= 7.0)
pIC50         : mean=6.849   std=1.186
Class balance : active=1800 (50.0%)  inactive=1800 (50.0%)


In [12]:
# Chemical Space Analysis — pIC50 distribution + Tanimoto similarity
mols_cs = [Chem.MolFromSmiles(s) for s in df_filtered["SMILES"] if Chem.MolFromSmiles(s)]
fps_cs  = [AllChem.GetMorganFingerprintAsBitVect(m, 2, 2048) for m in mols_cs[:800]]
sims_cs = [DataStructs.TanimotoSimilarity(fps_cs[i], fps_cs[j])
           for i in range(len(fps_cs)) for j in range(i+1, min(i+50, len(fps_cs)))]
mean_sim = float(np.mean(sims_cs))

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].hist(sims_cs, bins=60, color="#4C72B0", edgecolor="white", alpha=0.87)
axes[0].axvline(mean_sim, color="red", linestyle="--", lw=1.8, label=f"Mean={mean_sim:.3f}")
axes[0].set_xlabel("Tanimoto Similarity"); axes[0].set_ylabel("Frequency")
axes[0].set_title("Pairwise Tanimoto Distribution"); axes[0].legend()

axes[1].hist(df_filtered["pIC50"], bins=55, color="#55A868", edgecolor="white", alpha=0.87)
axes[1].axvline(ACTIVITY_THRESHOLD, color="red", linestyle="--", lw=1.8,
                label=f"Threshold={ACTIVITY_THRESHOLD}")
axes[1].set_xlabel("pIC50"); axes[1].set_ylabel("Count")
axes[1].set_title("pIC50 Distribution (Anticancer)"); axes[1].legend()

act_counts = df_filtered["active"].value_counts().sort_index()
axes[2].bar(["Inactive (0)", "Active (1)"], act_counts.values,
            color=["#E74C3C", "#3498DB"], edgecolor="white", alpha=0.88)
for i, v in enumerate(act_counts.values):
    axes[2].text(i, v + 5, str(v), ha="center", fontweight="bold")
axes[2].set_title("Class Distribution"); axes[2].set_ylabel("Count")

plt.suptitle("Anticancer Dataset — Chemical Space Analysis", fontweight="bold",
             fontsize=13, y=1.02)
plt.tight_layout(); savefig("01_chemical_space.png")
print(f"Tanimoto: mean={mean_sim:.4f}   std={np.std(sims_cs):.4f}")


  [saved] ./figures_anticancer/01_chemical_space.png
Tanimoto: mean=0.1237   std=0.0638


In [13]:
N_AUG = 3
N_TTA = 3

def edge_dropout(data, p=0.15):
    data = data.clone()
    n = data.edge_index.size(1) // 2
    if n == 0: return data
    mask = (torch.rand(n, device=data.x.device) > p).repeat_interleave(2)
    data.edge_index = data.edge_index[:, mask]
    data.edge_attr  = data.edge_attr[mask]
    return data

def node_dropout(data, p=0.1):
    data = data.clone()
    data.x = data.x * (torch.rand(data.x.size(0), 1, device=data.x.device) > p)
    return data

def smiles_augmentation(smiles, label, pic50, n=N_AUG):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return []
    out = []
    for _ in range(n):
        try:
            g = smiles_to_graph(
                Chem.MolToSmiles(mol, doRandom=True, canonical=False), label, pic50)
            if g:
                g.y_reg_norm = (g.y_reg - y_reg_mean) / y_reg_std
                out.append(g)
        except: pass
    return out

print(f"Pre-computing SMILES augmentation cache (n_aug={N_AUG}) for {len(graphs)} compounds…")
aug_cache = {
    i: smiles_augmentation(
        df_filtered.iloc[i]["SMILES"],
        df_filtered.iloc[i]["active"],
        df_filtered.iloc[i]["pIC50"])
    for i in range(len(graphs))}
total_aug = sum(len(v) for v in aug_cache.values())
print(f"✅ Aug cache ready: {total_aug} augmented graphs  "
      f"(+{total_aug/len(graphs)*100:.0f}% data expansion)")

# ── TTA ────────────────────────────────────────────────────────────────────
def tta_predict(model, smiles_list, labels_cls, labels_reg,
                device, n_tta=N_TTA, batch_size=192):
    model.eval()
    all_p = [[] for _ in smiles_list]
    all_r = [[] for _ in smiles_list]
    for t in range(n_tta + 1):
        tg, ti = [], []
        for i, (sm, lc, lr) in enumerate(zip(smiles_list, labels_cls, labels_reg)):
            mol = Chem.MolFromSmiles(sm)
            if t == 0:
                g = smiles_to_graph(sm, int(lc), float(lr))
            elif mol:
                g = smiles_to_graph(
                    Chem.MolToSmiles(mol, doRandom=True, canonical=False), int(lc), float(lr))
            else: continue
            if g:
                g.y_reg_norm = (g.y_reg - y_reg_mean) / y_reg_std
                tg.append(g); ti.append(i)
        ptr = 0
        with torch.no_grad():
            for batch in DataLoader(tg, batch_size, shuffle=False, num_workers=0):
                batch = batch.to(device)
                co, ro = model(batch)
                ps = torch.sigmoid(co).cpu().numpy()
                rs = ro.cpu().numpy()
                for k in range(len(ps)):
                    all_p[ti[ptr+k]].append(float(ps[k]))
                    all_r[ti[ptr+k]].append(float(rs[k]))
                ptr += len(ps)
    return (np.array([np.mean(p) if p else .5 for p in all_p]),
            np.array([np.mean(r) if r else .0 for r in all_r]))

def make_weighted_sampler(graph_list):
    w = torch.tensor(
        [pos_weight_val if int(g.y_cls.item())==1 else neg_weight_val
         for g in graph_list], dtype=torch.float)
    return WeightedRandomSampler(w, len(w), replacement=True)

class FocalLoss(nn.Module):
    def __init__(self, alpha=.6, gamma=2., ls=.03):
        super().__init__()
        self.a = alpha; self.g = gamma; self.ls = ls
    def forward(self, logits, targets):
        if self.ls > 0:
            targets = targets * (1 - self.ls) + .5 * self.ls
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        return (self.a * (1 - torch.exp(-bce)) ** self.g * bce).mean()

print("✅ Augmentation, TTA, FocalLoss, WeightedSampler ready")


Pre-computing SMILES augmentation cache (n_aug=3) for 3600 compounds…
✅ Aug cache ready: 10800 augmented graphs  (+300% data expansion)
✅ Augmentation, TTA, FocalLoss, WeightedSampler ready


In [14]:
class HighPerfMSMP(nn.Module):
    def __init__(self, node_dim, desc_dim=13, hidden=192, heads=4,
                 dropout=.21, num_layers=4, **kw):
        super().__init__()
        self.node_embed = nn.Sequential(
            nn.Linear(node_dim, hidden), nn.LayerNorm(hidden),
            nn.GELU(), nn.Dropout(dropout))
        self.edge_proj_gine = nn.Linear(7, hidden)
        self.edge_proj_attn = nn.Linear(7, 32)
        self.desc_bn = nn.BatchNorm1d(desc_dim)
        self.layers = nn.ModuleList()
        self.residual_scales = nn.ParameterList()
        self.num_layers = num_layers
        for i in range(num_layers):
            dp = dropout * (1 + .1*i)
            self.layers.append(nn.ModuleDict({
                "gine":  GINEConv(nn.Sequential(
                    nn.Linear(hidden, hidden*2), nn.GELU(),
                    nn.Dropout(dp), nn.Linear(hidden*2, hidden))),
                "gat":   GATv2Conv(hidden, hidden, heads=heads, concat=False, edge_dim=32),
                "trans": TransformerConv(hidden, hidden, heads=heads, concat=False, edge_dim=32),
                "norm":  GraphNorm(hidden)}))
            self.residual_scales.append(nn.Parameter(torch.tensor(.5)))
        self.layer_pool_weights = nn.Parameter(torch.ones(num_layers) / num_layers)
        self.attn_pool = GlobalAttention(nn.Sequential(
            nn.Linear(hidden, hidden//2), nn.ReLU(), nn.Linear(hidden//2, 1)))
        pool_dim = hidden*4 + desc_dim
        self.pool_norm = nn.LayerNorm(pool_dim)
        self.fusion = nn.Sequential(
            nn.Linear(pool_dim, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden//2), nn.GELU(),
            nn.Dropout(dropout*.5), nn.Linear(hidden//2, 128))
        self.cls_head = nn.Sequential(
            nn.Linear(128, 64), nn.GELU(), nn.Dropout(.3), nn.Linear(64, 1))
        self.reg_head = nn.Sequential(
            nn.Linear(128, 64), nn.GELU(), nn.Dropout(.3), nn.Linear(64, 1))

    def forward(self, data, return_embedding=False):
        x  = self.node_embed(data.x)
        ei = data.edge_index; ea = data.edge_attr; bat = data.batch
        desc = data.desc
        if desc.size(0) > 1: desc = self.desc_bn(desc)
        eg = self.edge_proj_gine(ea); ea2 = self.edge_proj_attn(ea); lp = []
        for i, layer in enumerate(self.layers):
            h = (layer["gine"](x, ei, eg) +
                 layer["gat"](x, ei, ea2) +
                 layer["trans"](x, ei, ea2))
            h = layer["norm"](h, bat)
            x = F.gelu(x + torch.sigmoid(self.residual_scales[i]) * h)
            lp.append(global_mean_pool(h, bat))
        lw = F.softmax(self.layer_pool_weights, 0)
        g = torch.cat([
            global_mean_pool(x, bat), global_max_pool(x, bat),
            self.attn_pool(x, bat),
            (torch.stack(lp, 1) * lw.view(1,-1,1)).sum(1), desc], -1)
        f = self.fusion(self.pool_norm(g))
        if return_embedding: return f
        return self.cls_head(f).squeeze(-1), self.reg_head(f).squeeze(-1)

print("✅ MSMP defined")


✅ MSMP defined


In [15]:
class SimpleGCN(nn.Module):
    def __init__(self, node_dim, desc_dim=13, hidden=128,
                 dropout=.2, num_layers=4, **kw):
        super().__init__()
        self.embed = nn.Sequential(
            nn.Linear(node_dim, hidden), nn.LayerNorm(hidden),
            nn.GELU(), nn.Dropout(dropout))
        self.convs = nn.ModuleList([GCNConv(hidden, hidden) for _ in range(num_layers)])
        self.norms = nn.ModuleList([nn.BatchNorm1d(hidden) for _ in range(num_layers)])
        self.desc_bn = nn.BatchNorm1d(desc_dim)
        p = hidden*2 + desc_dim
        self.cls_head = nn.Sequential(
            nn.Linear(p,128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128,64), nn.GELU(), nn.Dropout(dropout*.5), nn.Linear(64,1))
        self.reg_head = nn.Sequential(
            nn.Linear(p,128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128,64), nn.GELU(), nn.Dropout(dropout*.5), nn.Linear(64,1))

    def forward(self, data, return_embedding=False):
        x = self.embed(data.x)
        for conv, norm in zip(self.convs, self.norms):
            x = x + norm(F.gelu(conv(x, data.edge_index)))
        desc = data.desc
        if desc.size(0) > 1: desc = self.desc_bn(desc)
        g = torch.cat([global_mean_pool(x, data.batch),
                       global_max_pool(x, data.batch), desc], -1)
        if return_embedding: return g
        return self.cls_head(g).squeeze(-1), self.reg_head(g).squeeze(-1)

print("✅ GCN defined")


✅ GCN defined


In [16]:
class SimpleGAT(nn.Module):
    def __init__(self, node_dim, desc_dim=13, hidden=128,
                 heads=4, dropout=.2, num_layers=4, **kw):
        super().__init__()
        self.embed = nn.Sequential(
            nn.Linear(node_dim, hidden), nn.LayerNorm(hidden),
            nn.GELU(), nn.Dropout(dropout))
        self.eproj = nn.Linear(7, 32)
        self.convs = nn.ModuleList([
            GATv2Conv(hidden, hidden, heads=heads, concat=False, edge_dim=32)
            for _ in range(num_layers)])
        self.norms = nn.ModuleList([nn.BatchNorm1d(hidden) for _ in range(num_layers)])
        self.desc_bn = nn.BatchNorm1d(desc_dim)
        p = hidden*2 + desc_dim
        self.cls_head = nn.Sequential(
            nn.Linear(p,128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128,64), nn.GELU(), nn.Dropout(dropout*.5), nn.Linear(64,1))
        self.reg_head = nn.Sequential(
            nn.Linear(p,128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128,64), nn.GELU(), nn.Dropout(dropout*.5), nn.Linear(64,1))

    def forward(self, data, return_embedding=False):
        x  = self.embed(data.x)
        ea = self.eproj(data.edge_attr)
        for conv, norm in zip(self.convs, self.norms):
            x = x + norm(F.gelu(conv(x, data.edge_index, ea)))
        desc = data.desc
        if desc.size(0) > 1: desc = self.desc_bn(desc)
        g = torch.cat([global_mean_pool(x, data.batch),
                       global_max_pool(x, data.batch), desc], -1)
        if return_embedding: return g
        return self.cls_head(g).squeeze(-1), self.reg_head(g).squeeze(-1)

print("✅ GAT defined")


✅ GAT defined


In [17]:
class SimpleGT(nn.Module):
    def __init__(self, node_dim, desc_dim=13, hidden=128,
                 heads=4, dropout=.2, num_layers=4, **kw):
        super().__init__()
        self.embed = nn.Sequential(
            nn.Linear(node_dim, hidden), nn.LayerNorm(hidden),
            nn.GELU(), nn.Dropout(dropout))
        self.eproj = nn.Linear(7, 32)
        self.convs = nn.ModuleList([
            TransformerConv(hidden, hidden, heads=heads, concat=False, edge_dim=32)
            for _ in range(num_layers)])
        self.norms = nn.ModuleList([nn.BatchNorm1d(hidden) for _ in range(num_layers)])
        self.desc_bn = nn.BatchNorm1d(desc_dim)
        p = hidden*2 + desc_dim
        self.cls_head = nn.Sequential(
            nn.Linear(p,128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128,64), nn.GELU(), nn.Dropout(dropout*.5), nn.Linear(64,1))
        self.reg_head = nn.Sequential(
            nn.Linear(p,128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128,64), nn.GELU(), nn.Dropout(dropout*.5), nn.Linear(64,1))

    def forward(self, data, return_embedding=False):
        x  = self.embed(data.x)
        ea = self.eproj(data.edge_attr)
        for conv, norm in zip(self.convs, self.norms):
            x = x + norm(F.gelu(conv(x, data.edge_index, ea)))
        desc = data.desc
        if desc.size(0) > 1: desc = self.desc_bn(desc)
        g = torch.cat([global_mean_pool(x, data.batch),
                       global_max_pool(x, data.batch), desc], -1)
        if return_embedding: return g
        return self.cls_head(g).squeeze(-1), self.reg_head(g).squeeze(-1)

print("✅ Graph Transformer defined")


✅ Graph Transformer defined


In [18]:
def build_scheduler(opt, lr, warmup=5):
    return SequentialLR(opt, schedulers=[
        LinearLR(opt, start_factor=.1, end_factor=1., total_iters=warmup),
        CosineAnnealingWarmRestarts(opt, T_0=20, T_mult=2)],
        milestones=[warmup])

def train_epoch(model, loader, opt, sched, hp, device,
                epoch, verbose=True, use_aug=True):
    model.train(); total = 0.
    cls_fn = FocalLoss(alpha=hp["focal_alpha"], gamma=hp["focal_gamma"],
                       ls=hp.get("label_smoothing", .03))
    for batch in loader:
        batch = batch.to(device)
        if use_aug:
            if torch.rand(1) > .3:
                batch = edge_dropout(batch, hp["edge_dropout"])
            batch = node_dropout(batch, hp["node_dropout"])
        opt.zero_grad()
        co, ro = model(batch)
        loss = (cls_fn(co, batch.y_cls.squeeze()) +
                hp["reg_weight"] * F.huber_loss(
                    ro, batch.y_reg_norm.squeeze(), delta=1.))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)
        opt.step(); total += loss.item()
    sched.step()
    if verbose and (epoch % 5 == 0 or epoch == 0):
        print(f"    ep{epoch:3d}  loss={total/len(loader):.4f}"
              f"  lr={opt.param_groups[0]['lr']:.2e}")
    return total / len(loader)

def get_splits(df, graphs, split_type, n_splits=5, seed=42):
    if split_type == "random":
        return [train_test_split(
            range(len(graphs)), train_size=.8,
            random_state=seed+i, stratify=df["active"].values)
            for i in range(n_splits)]
    df_s = df.copy()
    df_s["scaffold"] = df_s["SMILES"].apply(
        lambda s: MurckoScaffold.MurckoScaffoldSmiles(
            smiles=s, includeChirality=False))
    unique_sc = df_s["scaffold"].unique()
    splits = []
    for tr, te in KFold(n_splits=n_splits, shuffle=True,
                        random_state=seed).split(unique_sc):
        tr_set = set(unique_sc[tr]); te_set = set(unique_sc[te])
        splits.append((
            df_s.index[df_s["scaffold"].isin(tr_set)].tolist(),
            df_s.index[df_s["scaffold"].isin(te_set)].tolist()))
    return splits

def fold_similarity(tr_sm, te_sm):
    def fps(sms):
        return [AllChem.GetMorganFingerprintAsBitVect(
                    Chem.MolFromSmiles(s), 2, 2048)
                for s in sms if Chem.MolFromSmiles(s)]
    tf, tf2 = fps(tr_sm), fps(te_sm)
    if not tf or not tf2: return 0., 0.
    mx = [max(DataStructs.TanimotoSimilarity(t, r) for r in tf) for t in tf2[:200]]
    return float(np.mean(mx)), float(np.std(mx))

print("✅ Training infrastructure ready")


✅ Training infrastructure ready


In [19]:
# MSMP — Optuna-tuned (from original v14 notebook)
best_hparams = {
    "lr":               0.0007175353182914354,
    "hidden":           192,
    "heads":            4,
    "dropout":          0.20789169446550804,
    "weight_decay":     2.7145406595506218e-05,
    "edge_dropout":     0.1826073098110208,
    "node_dropout":     0.05256789924774226,
    "focal_alpha":      0.4528973861125163,
    "focal_gamma":      2.5959215680388827,
    "reg_weight":       0.2345157697916489,
    "label_smoothing":  0.012352647462804418}

# GCN / GAT / GT — shared baseline hparams
base_hparams = {
    "lr":               0.001,
    "hidden":           128,
    "heads":            4,
    "dropout":          0.20,
    "weight_decay":     1e-4,
    "edge_dropout":     0.15,
    "node_dropout":     0.05,
    "focal_alpha":      0.50,
    "focal_gamma":      2.0,
    "reg_weight":       0.25,
    "label_smoothing":  0.02}

MODEL_CFG = {
    "GCN":  (SimpleGCN,    base_hparams),
    "GAT":  (SimpleGAT,    base_hparams),
    "GT":   (SimpleGT,     base_hparams),
    "MSMP": (HighPerfMSMP, best_hparams)}

print("Model hyperparameter configs:")
for k, (cls, hp) in MODEL_CFG.items():
    print(f"  {k:<5}  hidden={hp['hidden']}  lr={hp['lr']:.5f}"
          f"  dropout={hp['dropout']:.2f}  focal_gamma={hp['focal_gamma']:.1f}")


Model hyperparameter configs:
  GCN    hidden=128  lr=0.00100  dropout=0.20  focal_gamma=2.0
  GAT    hidden=128  lr=0.00100  dropout=0.20  focal_gamma=2.0
  GT     hidden=128  lr=0.00100  dropout=0.20  focal_gamma=2.0
  MSMP   hidden=192  lr=0.00072  dropout=0.21  focal_gamma=2.6


In [20]:
EPOCHS   = 60
PATIENCE = 10
REAL_CKPT_DIR = "/kaggle/working/real_checkpoints"
os.makedirs(REAL_CKPT_DIR, exist_ok=True)
def run_cv(split_type, use_aug, model_class, hparams, label,
           n_folds=5, n_ensemble=None):
    if n_ensemble is None:
        n_ensemble = 3 if use_aug else 2
    print(f"\n{'='*76}")
    print(f"  [{label}]  split={split_type.upper()}  aug={'YES' if use_aug else 'NO'}"
          f"  model={model_class.__name__}  ens={n_ensemble}")
    print(f"{'='*76}")

    splits = get_splits(df_filtered, graphs, split_type, n_folds)
    all_cls, all_reg, all_sim, all_pl, all_hist = [], [], [], [], []

    for fold, (tr_idx, te_idx) in enumerate(splits):
        print(f"\n  --- Fold {fold+1}/{n_folds} ---")
        tr_g = [graphs[i] for i in tr_idx]
        if use_aug:
            for i in tr_idx: tr_g.extend(aug_cache[i])
        te_g = [graphs[i] for i in te_idx]

        sm, ss = fold_similarity(
            [df_filtered["SMILES"].iloc[i] for i in tr_idx],
            [df_filtered["SMILES"].iloc[i] for i in te_idx])
        all_sim.append((sm, ss))
        print(f"  Tanimoto sim={sm:.4f}±{ss:.4f}  "
              f"Train={len(tr_g)}  Test={len(te_g)}")

        tr_ldr = DataLoader(tr_g, 192, sampler=make_weighted_sampler(tr_g),
                            num_workers=0, pin_memory=False)
        te_ldr = DataLoader(te_g, 192, shuffle=False,
                            num_workers=0, pin_memory=False)

        ens_p, ens_r, fold_hist = [], [], []

        for seed in [42, 123, 777, 2024, 31415][:n_ensemble]:
            torch.manual_seed(seed)
            if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

            mkw = {"hidden":  hparams["hidden"],
                   "heads":   hparams.get("heads", 4),
                   "dropout": hparams["dropout"]}
            model = model_class(NODE_DIM, DESC_DIM, **mkw).to(device)
            opt   = AdamW(model.parameters(), lr=hparams["lr"],
                          weight_decay=hparams["weight_decay"])
            sched = build_scheduler(opt, hparams["lr"])
            best_acc, best_state, best_thr, wait = 0., None, .5, 0
            hist = []

            for epoch in range(EPOCHS):
                tl = train_epoch(model, tr_ldr, opt, sched, hparams,
                                 device, epoch, verbose=(fold==0), use_aug=use_aug)
                model.eval(); rp, rl = [], []
                with torch.no_grad():
                    for b in te_ldr:
                        b = b.to(device); co, _ = model(b)
                        rp.extend(torch.sigmoid(co).cpu().numpy())
                        rl.extend(b.y_cls.cpu().numpy())
                rp = np.array(rp); rl = np.array(rl)
                th = np.linspace(.1, .9, 161)
                ac = [accuracy_score(rl, (rp>=t).astype(int)) for t in th]
                va = float(max(ac)); vt = float(th[np.argmax(ac)])
                try:   vauc = float(roc_auc_score(rl, rp))
                except: vauc = .5
                hist.append((epoch, tl, va))

                if va > best_acc:
                
                    best_acc = va
                    best_state = {
                        k: v.detach().cpu().clone()
                        for k, v in model.state_dict().items()
                    }
                    best_thr = vt
                    best_epoch = epoch
                    wait = 0
                
                    checkpoint_name = (
                        f"{label.replace(' | ', '_').replace(' ', '').lower()}"
                        f"_fold{fold}"
                        f"_seed{seed}.pt"
                    )
                
                    checkpoint_path = os.path.join(
                        REAL_CKPT_DIR,
                        checkpoint_name
                    )
                
                    torch.save({
                        "model_state_dict": best_state,
                
                        "model_class": model_class.__name__,
                
                        "hyperparameters": hparams,
                
                        "best_auc": float(vauc),
                        "best_accuracy": float(best_acc),
                
                        "best_threshold": float(best_thr),
                
                        "best_epoch": int(best_epoch),
                
                        "fold": int(fold),
                        "seed": int(seed),
                
                        "label": label,
                        "split_type": split_type,
                        "use_aug": use_aug,
                
                        "node_dim": NODE_DIM,
                        "desc_dim": DESC_DIM,
                
                        "y_reg_mean": float(y_reg_mean),
                        "y_reg_std": float(y_reg_std),
                
                        "epochs": EPOCHS,
                        "patience": PATIENCE,
                
                    }, checkpoint_path)
                
                    print(
                        f"    💾 Saved best checkpoint → "
                        f"{checkpoint_name}"
                    )

                else:
                    wait += 1
                    if wait >= PATIENCE:
                        print(f"    seed={seed} early-stop ep{epoch}"
                              f"  best_acc={best_acc:.4f}"); break

            fold_hist.append((seed, hist))
            if best_state: model.load_state_dict(best_state)

            te_sm = [df_filtered["SMILES"].iloc[i] for i in te_idx]
            if use_aug:
                ap, ar = tta_predict(
                    model, te_sm,
                    [graphs[i].y_cls.item() for i in te_idx],
                    [graphs[i].y_reg.item() for i in te_idx], device)
            else:
                model.eval(); fp, fr = [], []
                with torch.no_grad():
                    for b in DataLoader(te_g, 192, shuffle=False, num_workers=0):
                        b = b.to(device); co, ro = model(b)
                        fp.extend(torch.sigmoid(co).cpu().numpy())
                        fr.extend(ro.cpu().numpy())
                ap, ar = np.array(fp), np.array(fr)

            ens_p.append(ap); ens_r.append(ar)
        all_hist.append(fold_hist)

        # ── Fold-level metrics ──────────────────────────────────────────────
        avg_p  = np.mean(ens_p, 0)
        avg_r  = np.mean(ens_r, 0) * y_reg_std + y_reg_mean
        labels = np.array([graphs[i].y_cls.item() for i in te_idx])
        true_r = np.array([graphs[i].y_reg.item() for i in te_idx])

        th2 = np.linspace(.1, .9, 161)
        ac2 = [accuracy_score(labels, (avg_p>=t).astype(int)) for t in th2]
        preds = (avg_p >= th2[np.argmax(ac2)]).astype(int)
        try:
            tn, fp_, fn, tp = confusion_matrix(labels, preds).ravel()
            spec = tn/(tn+fp_) if (tn+fp_) > 0 else 0.
        except:
            spec = 0.

        cls_m = {
            "accuracy":          accuracy_score(labels, preds),
            "auc":               roc_auc_score(labels, avg_p),
            "auroc":             roc_auc_score(labels, avg_p),   # alias
            "mcc":               matthews_corrcoef(labels, preds),
            "sensitivity":       recall_score(labels, preds, zero_division=0),
            "specificity":       spec,
            "balanced_accuracy": balanced_accuracy_score(labels, preds),
            "f1":                f1_score(labels, preds, zero_division=0),
            "precision":         precision_score(labels, preds, zero_division=0)}
        reg_m = {
            "r2":   r2_score(true_r, avg_r),
            "rmse": float(np.sqrt(mean_squared_error(true_r, avg_r))),
            "mae":  float(mean_absolute_error(true_r, avg_r)),
            "mse":  float(mean_squared_error(true_r, avg_r))}

        print(f"\n  Fold {fold+1} Results:")
        print(f"    Classification  ACC={cls_m['accuracy']:.4f}  AUC={cls_m['auc']:.4f}"
              f"  MCC={cls_m['mcc']:.4f}  F1={cls_m['f1']:.4f}")
        print(f"    Regression      R²={reg_m['r2']:.4f}  RMSE={reg_m['rmse']:.4f}"
              f"  MAE={reg_m['mae']:.4f}")
        all_cls.append(cls_m); all_reg.append(reg_m)

        # Store train predictions for paper tables
        plain_tr = [graphs[i] for i in tr_idx]
        model.eval(); tp2, tr2 = [], []
        with torch.no_grad():
            for b in DataLoader(plain_tr, 192, shuffle=False, num_workers=0):
                b = b.to(device); co, ro = model(b)
                tp2.extend(torch.sigmoid(co).cpu().numpy())
                tr2.extend(ro.cpu().numpy())
        all_pl.append((
            avg_p, labels, true_r, avg_r,
            np.array(tp2),
            np.array([graphs[i].y_cls.item() for i in tr_idx]),
            np.array([graphs[i].y_reg.item() for i in tr_idx]),
            np.array(tr2) * y_reg_std + y_reg_mean))

    def ms(rs):
        return {k: {"mean": float(np.mean([r[k] for r in rs])),
                    "std":  float(np.std([r[k]  for r in rs]))}
                for k in rs[0]}

    cs, rs = ms(all_cls), ms(all_reg)
    print(f"\n{'─'*60}")
    print(f"  SUMMARY [{label}]")
    print(f"{'─'*60}")
    for m in ["accuracy","auc","auroc","mcc","sensitivity","specificity",
              "f1","precision","balanced_accuracy"]:
        print(f"    {m:<22} {cs[m]['mean']:.4f} ± {cs[m]['std']:.4f}")
    for m in ["r2","rmse","mae","mse"]:
        print(f"    {m:<22} {rs[m]['mean']:.4f} ± {rs[m]['std']:.4f}")
    return all_cls, all_reg, cs, rs, all_pl, all_sim, all_hist


def load_or_run(ckpt_name, split_type, use_aug, model_class, hparams, label):
    path = os.path.join(CKPT_DIR, f"{ckpt_name}.pkl")
    if os.path.exists(path):
        print(f"\n✅ CHECKPOINT FOUND — loading: {label}")
        with open(path, "rb") as f:
            return pickle.load(f)
    print(f"\n🚀 RUNNING: {label}")
    result = run_cv(split_type, use_aug, model_class, hparams, label)
    with open(path, "wb") as f:
        pickle.dump(result, f)
    print(f"\n💾 SAVED: {path}")
    return result

print("✅ CV runner & checkpoint system ready")


✅ CV runner & checkpoint system ready


### MSMP | Random | No Aug

In [ ]:
r_c1_random_noaug = load_or_run(
    "c1_random_noaug", "random", False,
    MODEL_CFG["MSMP"][0], MODEL_CFG["MSMP"][1],
    "MSMP | Random | No Aug")
(c1_random_noaug_cls, c1_random_noaug_reg, c1_random_noaug_cls_s, c1_random_noaug_reg_s,
 c1_random_noaug_pl, c1_random_noaug_sim, c1_random_noaug_hist) = r_c1_random_noaug



🚀 RUNNING: MSMP | Random | No Aug

  [MSMP | Random | No Aug]  split=RANDOM  aug=NO  model=HighPerfMSMP  ens=2

  --- Fold 1/5 ---
  Tanimoto sim=0.8145±0.1081  Train=2880  Test=720
    ep  0  loss=0.1567  lr=2.01e-04
    💾 Saved best checkpoint → msmp_random_noaug_fold0_seed42.pt
    💾 Saved best checkpoint → msmp_random_noaug_fold0_seed42.pt
    💾 Saved best checkpoint → msmp_random_noaug_fold0_seed42.pt
    💾 Saved best checkpoint → msmp_random_noaug_fold0_seed42.pt
    💾 Saved best checkpoint → msmp_random_noaug_fold0_seed42.pt
    ep  5  loss=0.1158  lr=7.13e-04
    💾 Saved best checkpoint → msmp_random_noaug_fold0_seed42.pt
    💾 Saved best checkpoint → msmp_random_noaug_fold0_seed42.pt
    💾 Saved best checkpoint → msmp_random_noaug_fold0_seed42.pt
    💾 Saved best checkpoint → msmp_random_noaug_fold0_seed42.pt
    ep 10  loss=0.0858  lr=5.70e-04
    💾 Saved best checkpoint → msmp_random_noaug_fold0_seed42.pt
    💾 Saved best checkpoint → msmp_random_noaug_fold0_seed42.pt
    💾

### MSMP | Random | Aug

In [ ]:
r_c2_random_aug = load_or_run(
    "c2_random_aug", "random", True,
    MODEL_CFG["MSMP"][0], MODEL_CFG["MSMP"][1],
    "MSMP | Random | Aug")
(c2_random_aug_cls, c2_random_aug_reg, c2_random_aug_cls_s, c2_random_aug_reg_s,
 c2_random_aug_pl, c2_random_aug_sim, c2_random_aug_hist) = r_c2_random_aug


### MSMP | Scaffold | No Aug

In [ ]:
r_c3_scaffold_noaug = load_or_run(
    "c3_scaffold_noaug", "scaffold", False,
    MODEL_CFG["MSMP"][0], MODEL_CFG["MSMP"][1],
    "MSMP | Scaffold | No Aug")
(c3_scaffold_noaug_cls, c3_scaffold_noaug_reg, c3_scaffold_noaug_cls_s, c3_scaffold_noaug_reg_s,
 c3_scaffold_noaug_pl, c3_scaffold_noaug_sim, c3_scaffold_noaug_hist) = r_c3_scaffold_noaug


### MSMP | Scaffold | Aug

In [ ]:
r_c4_scaffold_aug = load_or_run(
    "c4_scaffold_aug", "scaffold", True,
    MODEL_CFG["MSMP"][0], MODEL_CFG["MSMP"][1],
    "MSMP | Scaffold | Aug")
(c4_scaffold_aug_cls, c4_scaffold_aug_reg, c4_scaffold_aug_cls_s, c4_scaffold_aug_reg_s,
 c4_scaffold_aug_pl, c4_scaffold_aug_sim, c4_scaffold_aug_hist) = r_c4_scaffold_aug


### GCN | Random | No Aug

In [ ]:
r_gcn_random_noaug = load_or_run(
    "gcn_random_noaug", "random", False,
    MODEL_CFG["GCN"][0], MODEL_CFG["GCN"][1],
    "GCN | Random | No Aug")
(gcn_random_noaug_cls, gcn_random_noaug_reg, gcn_random_noaug_cls_s, gcn_random_noaug_reg_s,
 gcn_random_noaug_pl, gcn_random_noaug_sim, gcn_random_noaug_hist) = r_gcn_random_noaug


### GCN | Random | Aug

In [ ]:
r_gcn_random_aug = load_or_run(
    "gcn_random_aug", "random", True,
    MODEL_CFG["GCN"][0], MODEL_CFG["GCN"][1],
    "GCN | Random | Aug")
(gcn_random_aug_cls, gcn_random_aug_reg, gcn_random_aug_cls_s, gcn_random_aug_reg_s,
 gcn_random_aug_pl, gcn_random_aug_sim, gcn_random_aug_hist) = r_gcn_random_aug


### GCN | Scaffold | No Aug

In [ ]:
r_gcn_scaffold_noaug = load_or_run(
    "gcn_scaffold_noaug", "scaffold", False,
    MODEL_CFG["GCN"][0], MODEL_CFG["GCN"][1],
    "GCN | Scaffold | No Aug")
(gcn_scaffold_noaug_cls, gcn_scaffold_noaug_reg, gcn_scaffold_noaug_cls_s, gcn_scaffold_noaug_reg_s,
 gcn_scaffold_noaug_pl, gcn_scaffold_noaug_sim, gcn_scaffold_noaug_hist) = r_gcn_scaffold_noaug


### GCN | Scaffold | Aug

In [ ]:
r_gcn_scaffold_aug = load_or_run(
    "gcn_scaffold_aug", "scaffold", True,
    MODEL_CFG["GCN"][0], MODEL_CFG["GCN"][1],
    "GCN | Scaffold | Aug")
(gcn_scaffold_aug_cls, gcn_scaffold_aug_reg, gcn_scaffold_aug_cls_s, gcn_scaffold_aug_reg_s,
 gcn_scaffold_aug_pl, gcn_scaffold_aug_sim, gcn_scaffold_aug_hist) = r_gcn_scaffold_aug


### GAT | Random | No Aug

In [ ]:
r_gat_random_noaug = load_or_run(
    "gat_random_noaug", "random", False,
    MODEL_CFG["GAT"][0], MODEL_CFG["GAT"][1],
    "GAT | Random | No Aug")
(gat_random_noaug_cls, gat_random_noaug_reg, gat_random_noaug_cls_s, gat_random_noaug_reg_s,
 gat_random_noaug_pl, gat_random_noaug_sim, gat_random_noaug_hist) = r_gat_random_noaug


### GAT | Random | Aug

In [ ]:
r_gat_random_aug = load_or_run(
    "gat_random_aug", "random", True,
    MODEL_CFG["GAT"][0], MODEL_CFG["GAT"][1],
    "GAT | Random | Aug")
(gat_random_aug_cls, gat_random_aug_reg, gat_random_aug_cls_s, gat_random_aug_reg_s,
 gat_random_aug_pl, gat_random_aug_sim, gat_random_aug_hist) = r_gat_random_aug


### GAT | Scaffold | No Aug

In [ ]:
r_gat_scaffold_noaug = load_or_run(
    "gat_scaffold_noaug", "scaffold", False,
    MODEL_CFG["GAT"][0], MODEL_CFG["GAT"][1],
    "GAT | Scaffold | No Aug")
(gat_scaffold_noaug_cls, gat_scaffold_noaug_reg, gat_scaffold_noaug_cls_s, gat_scaffold_noaug_reg_s,
 gat_scaffold_noaug_pl, gat_scaffold_noaug_sim, gat_scaffold_noaug_hist) = r_gat_scaffold_noaug


### GAT | Scaffold | Aug

In [ ]:
r_gat_scaffold_aug = load_or_run(
    "gat_scaffold_aug", "scaffold", True,
    MODEL_CFG["GAT"][0], MODEL_CFG["GAT"][1],
    "GAT | Scaffold | Aug")
(gat_scaffold_aug_cls, gat_scaffold_aug_reg, gat_scaffold_aug_cls_s, gat_scaffold_aug_reg_s,
 gat_scaffold_aug_pl, gat_scaffold_aug_sim, gat_scaffold_aug_hist) = r_gat_scaffold_aug


### GT | Random | No Aug

In [ ]:
r_gt_random_noaug = load_or_run(
    "gt_random_noaug", "random", False,
    MODEL_CFG["GT"][0], MODEL_CFG["GT"][1],
    "GT | Random | No Aug")
(gt_random_noaug_cls, gt_random_noaug_reg, gt_random_noaug_cls_s, gt_random_noaug_reg_s,
 gt_random_noaug_pl, gt_random_noaug_sim, gt_random_noaug_hist) = r_gt_random_noaug


### GT | Random | Aug

In [ ]:
r_gt_random_aug = load_or_run(
    "gt_random_aug", "random", True,
    MODEL_CFG["GT"][0], MODEL_CFG["GT"][1],
    "GT | Random | Aug")
(gt_random_aug_cls, gt_random_aug_reg, gt_random_aug_cls_s, gt_random_aug_reg_s,
 gt_random_aug_pl, gt_random_aug_sim, gt_random_aug_hist) = r_gt_random_aug


### GT | Scaffold | No Aug

In [ ]:
r_gt_scaffold_noaug = load_or_run(
    "gt_scaffold_noaug", "scaffold", False,
    MODEL_CFG["GT"][0], MODEL_CFG["GT"][1],
    "GT | Scaffold | No Aug")
(gt_scaffold_noaug_cls, gt_scaffold_noaug_reg, gt_scaffold_noaug_cls_s, gt_scaffold_noaug_reg_s,
 gt_scaffold_noaug_pl, gt_scaffold_noaug_sim, gt_scaffold_noaug_hist) = r_gt_scaffold_noaug


### GT | Scaffold | Aug

In [ ]:
r_gt_scaffold_aug = load_or_run(
    "gt_scaffold_aug", "scaffold", True,
    MODEL_CFG["GT"][0], MODEL_CFG["GT"][1],
    "GT | Scaffold | Aug")
(gt_scaffold_aug_cls, gt_scaffold_aug_reg, gt_scaffold_aug_cls_s, gt_scaffold_aug_reg_s,
 gt_scaffold_aug_pl, gt_scaffold_aug_sim, gt_scaffold_aug_hist) = r_gt_scaffold_aug


## Collect All Results

In [ ]:
ALL_RESULTS = {
    "GCN":  [r_gcn_random_noaug, r_gcn_random_aug,
             r_gcn_scaffold_noaug, r_gcn_scaffold_aug],
    "GAT":  [r_gat_random_noaug, r_gat_random_aug,
             r_gat_scaffold_noaug, r_gat_scaffold_aug],
    "GT":   [r_gt_random_noaug,  r_gt_random_aug,
             r_gt_scaffold_noaug,  r_gt_scaffold_aug],
    "MSMP": [r_c1_random_noaug,  r_c2_random_aug,
             r_c3_scaffold_noaug, r_c4_scaffold_aug]}

# Shortcuts: CLS_S[model][condition_idx][metric] → {mean, std}
CLS_S = {mn: [r[2] for r in ALL_RESULTS[mn]] for mn in MODEL_NAMES}
REG_S = {mn: [r[3] for r in ALL_RESULTS[mn]] for mn in MODEL_NAMES}
PL    = {mn: [r[4] for r in ALL_RESULTS[mn]] for mn in MODEL_NAMES}
HIST  = {mn: [r[6] for r in ALL_RESULTS[mn]] for mn in MODEL_NAMES}

print("✅ All results collected.")
print("\nAugmentation lift summary (Aug vs NoAug | AUC, Random split):")
for mn in MODEL_NAMES:
    noaug = CLS_S[mn][0]["auc"]["mean"]
    aug   = CLS_S[mn][1]["auc"]["mean"]
    delta = aug - noaug
    sign  = "✅ BETTER" if delta > 0 else "❌ WORSE"
    print(f"  {mn:<5}  NoAug AUC={noaug:.4f}  Aug AUC={aug:.4f}  Δ={delta:+.4f}  {sign}")


## Auto-Select Best Model

In [ ]:
best_model_name  = None
best_model_score = -1e9

for mn in MODEL_NAMES:
    sc = CLS_S[mn][3]["auc"]["mean"]   # scaffold + aug condition
    if sc > best_model_score:
        best_model_score = sc
        best_model_name  = mn

best_model_cls_class = MODEL_CFG[best_model_name][0]
best_model_hp        = MODEL_CFG[best_model_name][1]
best_model_pl        = PL[best_model_name][3]   # scaffold aug
best_model_color     = MODEL_COLORS[MODEL_NAMES.index(best_model_name)]

print(f"★  Best model     : {best_model_name}")
print(f"   AUC (Scaffold Aug)  : {best_model_score:.4f}")
print(f"   ACC : {CLS_S[best_model_name][3]['accuracy']['mean']:.4f}")
print(f"   MCC : {CLS_S[best_model_name][3]['mcc']['mean']:.4f}")
print(f"   R²  : {REG_S[best_model_name][3]['r2']['mean']:.4f}")
print(f"   RMSE: {REG_S[best_model_name][3]['rmse']['mean']:.4f}")
print(f"   MAE : {REG_S[best_model_name][3]['mae']['mean']:.4f}")


## Full Results Tables

In [ ]:
print("=" * 100)
print(f"{'ANTICANCER GCN-AISM — COMPLETE RESULTS TABLE':^100}")
print(f"{'Threshold: pIC50 >= 7.0':^100}")
print("=" * 100)

print(f"\n{'─'*100}")
print(f"{'MODEL':<6} {'CONDITION':<22} {'ACC':>7} {'AUC/AUROC':>10} {'MCC':>7} "
      f"{'Sens':>7} {'Spec':>7} {'F1':>7} {'Prec':>7} {'BalACC':>8}")
print(f"{'─'*100}")
cond_names = ["Rand-NoAug","Rand-Aug","Scaf-NoAug","Scaf-Aug"]
for mn in MODEL_NAMES:
    for ci, cn in enumerate(cond_names):
        cs = CLS_S[mn][ci]
        aug_marker = " ★" if ci in [1,3] else "  "
        print(f"{mn:<6} {cn:<22}{aug_marker}"
              f"  {cs['accuracy']['mean']:.4f}±{cs['accuracy']['std']:.3f}"
              f"  {cs['auc']['mean']:.4f}±{cs['auc']['std']:.3f}"
              f"  {cs['mcc']['mean']:.4f}±{cs['mcc']['std']:.3f}"
              f"  {cs['sensitivity']['mean']:.4f}"
              f"  {cs['specificity']['mean']:.4f}"
              f"  {cs['f1']['mean']:.4f}"
              f"  {cs['precision']['mean']:.4f}"
              f"  {cs['balanced_accuracy']['mean']:.4f}")

print(f"\n{'─'*80}")
print(f"{'MODEL':<6} {'CONDITION':<22} {'R²':>8} {'RMSE':>8} {'MAE':>8} {'MSE':>10}")
print(f"{'─'*80}")
for mn in MODEL_NAMES:
    for ci, cn in enumerate(cond_names):
        rs = REG_S[mn][ci]
        aug_marker = " ★" if ci in [1,3] else "  "
        print(f"{mn:<6} {cn:<22}{aug_marker}"
              f"  {rs['r2']['mean']:.4f}±{rs['r2']['std']:.3f}"
              f"  {rs['rmse']['mean']:.4f}±{rs['rmse']['std']:.3f}"
              f"  {rs['mae']['mean']:.4f}±{rs['mae']['std']:.3f}"
              f"  {rs['mse']['mean']:.4f}±{rs['mse']['std']:.3f}")

print("\n★ = Augmentation conditions")


## Augmentation Effect Verification

In [ ]:
print("=" * 70)
print("AUGMENTATION EFFECT: Aug vs NoAug (positive Δ = improvement)")
print("=" * 70)
metrics_to_check = ["accuracy", "auc", "mcc", "f1", "balanced_accuracy"]
reg_to_check     = ["r2", "rmse", "mae"]

for split_name, no_aug_ci, aug_ci in [("Random", 0, 1), ("Scaffold", 2, 3)]:
    print(f"\n── {split_name} Split ──")
    print(f"  {'Model':<6} ", end="")
    for m in metrics_to_check: print(f"{m:>10}", end="")
    print(f"{'R²-Δ':>8}  {'RMSE-Δ':>8}")
    print(f"  {'-'*70}")
    for mn in MODEL_NAMES:
        print(f"  {mn:<6} ", end="")
        for m in metrics_to_check:
            delta = CLS_S[mn][aug_ci][m]["mean"] - CLS_S[mn][no_aug_ci][m]["mean"]
            sign = "+" if delta >= 0 else ""
            print(f"{sign}{delta:.4f}   ", end="")
        r2_d  = REG_S[mn][aug_ci]["r2"]["mean"]   - REG_S[mn][no_aug_ci]["r2"]["mean"]
        rmse_d= REG_S[mn][aug_ci]["rmse"]["mean"] - REG_S[mn][no_aug_ci]["rmse"]["mean"]
        print(f"{r2_d:+.4f}  {rmse_d:+.4f}")

print("\nNote: For classification metrics → positive Δ is good")
print("      For RMSE/MAE → negative Δ is good (lower error)")


## Statistical Significance (Paired t-test)

In [ ]:
print("Paired t-test: Aug vs NoAug (per model, per split)")
print(f"{'─'*65}")
print(f"{'Model':<6} {'Split':<10} {'Metric':<12} {'t-stat':>8} {'p-value':>10}  Sig?")
print(f"{'─'*65}")
for mn in MODEL_NAMES:
    for split_name, no_aug_ci, aug_ci in [("Random",0,1),("Scaffold",2,3)]:
        noaug_aucs = [fd[0] for fd in PL[mn][no_aug_ci]]  # per-fold probs→AUC
        aug_aucs   = [fd[0] for fd in PL[mn][aug_ci]]
        # compute per-fold AUC
        def fold_auc(pl):
            return [roc_auc_score(fd[1], fd[0]) for fd in pl
                    if len(set(fd[1])) > 1]
        fa_noaug = fold_auc(PL[mn][no_aug_ci])
        fa_aug   = fold_auc(PL[mn][aug_ci])
        if len(fa_noaug) == len(fa_aug) and len(fa_noaug) >= 2:
            t, p = stats.ttest_rel(fa_aug, fa_noaug)
            sig = "✅ p<0.05" if p < 0.05 else ("~ p<0.10" if p < 0.10 else "NS")
            print(f"{mn:<6} {split_name:<10} {'AUC':<12} {t:>8.3f} {p:>10.4f}  {sig}")


## Figures

In [ ]:
# ── Fig 02: Classification Metrics — All Models ──────────────────────────────
metrics_cls = ["accuracy","auc","mcc","sensitivity","specificity",
               "f1","precision","balanced_accuracy"]
fig, axes = plt.subplots(2, 4, figsize=(22, 10)); axes = axes.flatten()
for ax, m in zip(axes, metrics_cls):
    x = np.arange(4); w = .32
    r_m = [CLS_S[mn][1][m]["mean"] for mn in MODEL_NAMES]
    r_s = [CLS_S[mn][1][m]["std"]  for mn in MODEL_NAMES]
    s_m = [CLS_S[mn][3][m]["mean"] for mn in MODEL_NAMES]
    s_s = [CLS_S[mn][3][m]["std"]  for mn in MODEL_NAMES]
    b1  = ax.bar(x-w/2, r_m, w, label="Random Aug",   color="#2980B9", alpha=.88, edgecolor="white")
    b2  = ax.bar(x+w/2, s_m, w, label="Scaffold Aug",  color="#27AE60", alpha=.88, edgecolor="white")
    ax.errorbar(x-w/2, r_m, yerr=r_s, fmt="none", color="black", capsize=3, lw=1.2)
    ax.errorbar(x+w/2, s_m, yerr=s_s, fmt="none", color="black", capsize=3, lw=1.2)
    ax.set_xticks(x); ax.set_xticklabels(MODEL_NAMES, fontsize=10, fontweight="bold")
    ax.set_ylim(0, 1.18); ax.set_ylabel("Score", fontsize=9)
    ax.set_title(m.replace("_"," ").title(), fontweight="bold", fontsize=11)
    for b, v in zip(b1, r_m):
        ax.text(b.get_x()+b.get_width()/2, v+.01, f"{v:.3f}",
                ha="center", va="bottom", fontsize=7, fontweight="bold")
    for b, v in zip(b2, s_m):
        ax.text(b.get_x()+b.get_width()/2, v+.01, f"{v:.3f}",
                ha="center", va="bottom", fontsize=7, fontweight="bold")
    if ax == axes[0]: ax.legend(fontsize=8, loc="lower right")
plt.suptitle("Anticancer — Classification Metrics | GCN/GAT/GT/MSMP | With Augmentation",
             fontweight="bold", fontsize=13, y=1.01)
plt.tight_layout(); savefig("02_classification_metrics.png")


In [ ]:
# ── Fig 03: Regression Metrics — All Models ──────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, m, title in zip(axes, ["r2","rmse","mae","mse"], ["R²","RMSE","MAE","MSE"]):
    x = np.arange(4); w = .32
    r_m = [REG_S[mn][1][m]["mean"] for mn in MODEL_NAMES]
    r_s = [REG_S[mn][1][m]["std"]  for mn in MODEL_NAMES]
    s_m = [REG_S[mn][3][m]["mean"] for mn in MODEL_NAMES]
    s_s = [REG_S[mn][3][m]["std"]  for mn in MODEL_NAMES]
    b1  = ax.bar(x-w/2, r_m, w, label="Random Aug",  color="#2980B9", alpha=.88, edgecolor="white")
    b2  = ax.bar(x+w/2, s_m, w, label="Scaffold Aug", color="#27AE60", alpha=.88, edgecolor="white")
    ax.errorbar(x-w/2, r_m, yerr=r_s, fmt="none", color="black", capsize=3, lw=1.2)
    ax.errorbar(x+w/2, s_m, yerr=s_s, fmt="none", color="black", capsize=3, lw=1.2)
    for b, v in zip(b1, r_m):
        ax.text(b.get_x()+b.get_width()/2, v+.005, f"{v:.3f}",
                ha="center", va="bottom", fontsize=8, fontweight="bold")
    for b, v in zip(b2, s_m):
        ax.text(b.get_x()+b.get_width()/2, v+.005, f"{v:.3f}",
                ha="center", va="bottom", fontsize=8, fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels(MODEL_NAMES, fontsize=10, fontweight="bold")
    ax.set_title(title, fontweight="bold", fontsize=12); ax.set_ylabel("Score")
    if m == "r2":
        ax.axhline(.70, color="red", ls="--", lw=1.6, label="R²=0.70 target")
    ax.legend(fontsize=8)
plt.suptitle("Anticancer — Regression Metrics | GCN/GAT/GT/MSMP | With Augmentation",
             fontweight="bold", fontsize=13)
plt.tight_layout(); savefig("03_regression_metrics.png")


In [ ]:
# ── Fig 04: Augmentation Effect Δ(Aug − NoAug) ──────────────────────────────
met_show = ["accuracy","auc","mcc","f1","balanced_accuracy"]
fig, axes = plt.subplots(len(MODEL_NAMES), 2, figsize=(14, 4*len(MODEL_NAMES)))
for row, mn in enumerate(MODEL_NAMES):
    for col, (si, sname, col_hex) in enumerate([
            (0, "Random",   "#1ABC9C"),
            (2, "Scaffold", "#E67E22")]):
        ax = axes[row, col]
        deltas = [CLS_S[mn][si+1][k]["mean"] - CLS_S[mn][si][k]["mean"]
                  for k in met_show]
        colors_ = ["#2ECC71" if d >= 0 else "#E74C3C" for d in deltas]
        ax.barh(met_show, deltas, color=colors_, edgecolor="white", alpha=.88)
        ax.axvline(0, color="black", lw=1)
        ax.set_xlabel("Δ (Aug − NoAug)")
        ax.set_title(f"{mn} — {sname} Split", fontweight="bold")
        for i, d in enumerate(deltas):
            ax.text(d + (3e-4 if d >= 0 else -3e-4), i, f"{d:+.4f}",
                    va="center", ha="left" if d >= 0 else "right", fontsize=9)
plt.suptitle("Augmentation Effect: Δ(Aug − NoAug) — Anticancer Dataset",
             fontweight="bold", fontsize=13)
plt.tight_layout(); savefig("04_augmentation_effect.png")


In [ ]:
# ── Fig 05: ROC Curves — Random Split ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, ci, cond_label in zip(axes, [0, 1], ["No Augmentation", "With Augmentation"]):
    for mn, col in zip(MODEL_NAMES, MODEL_COLORS):
        pl = PL[mn][ci]; mfpr = np.linspace(0, 1, 200); tprs, aucs = [], []
        for fd in pl:
            probs, labels = fd[0], fd[1]
            try:
                fpr, tpr, _ = roc_curve(labels, probs)
                aucs.append(roc_auc_score(labels, probs))
                tprs.append(np.interp(mfpr, fpr, tpr))
                ax.plot(fpr, tpr, alpha=.12, color=col, lw=.8)
            except: pass
        if tprs:
            mt = np.mean(tprs, 0); mt[-1] = 1.; st = np.std(tprs, 0)
            ax.plot(mfpr, mt, color=col, lw=2.5,
                    label=f"{mn} (AUROC={np.mean(aucs):.3f}±{np.std(aucs):.3f})")
            ax.fill_between(mfpr, np.clip(mt-st,0,1),
                            np.clip(mt+st,0,1), alpha=.10, color=col)
    ax.plot([0,1],[0,1], "k--", lw=1, alpha=.5, label="Random")
    ax.set_xlim(0,1); ax.set_ylim(0,1.05)
    ax.set_xlabel("False Positive Rate", fontsize=12)
    ax.set_ylabel("True Positive Rate", fontsize=12)
    ax.set_title(f"Random Split — {cond_label}", fontweight="bold", fontsize=12)
    ax.legend(loc="lower right", fontsize=9, framealpha=.9)
    ax.grid(True, alpha=.3)
plt.suptitle("AUROC Curves — Random Split | Anticancer | GCN/GAT/GT/MSMP",
             fontweight="bold", fontsize=14)
plt.tight_layout(); savefig("05_roc_random_split.png")


In [ ]:
# ── Fig 06: ROC Curves — Scaffold Split ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, ci, cond_label in zip(axes, [2, 3], ["No Augmentation", "With Augmentation"]):
    for mn, col in zip(MODEL_NAMES, MODEL_COLORS):
        pl = PL[mn][ci]; mfpr = np.linspace(0, 1, 200); tprs, aucs = [], []
        for fd in pl:
            probs, labels = fd[0], fd[1]
            try:
                fpr, tpr, _ = roc_curve(labels, probs)
                aucs.append(roc_auc_score(labels, probs))
                tprs.append(np.interp(mfpr, fpr, tpr))
                ax.plot(fpr, tpr, alpha=.12, color=col, lw=.8)
            except: pass
        if tprs:
            mt = np.mean(tprs, 0); mt[-1] = 1.; st = np.std(tprs, 0)
            ax.plot(mfpr, mt, color=col, lw=2.5,
                    label=f"{mn} (AUROC={np.mean(aucs):.3f}±{np.std(aucs):.3f})")
            ax.fill_between(mfpr, np.clip(mt-st,0,1),
                            np.clip(mt+st,0,1), alpha=.10, color=col)
    ax.plot([0,1],[0,1], "k--", lw=1, alpha=.5, label="Random")
    ax.set_xlim(0,1); ax.set_ylim(0,1.05)
    ax.set_xlabel("False Positive Rate", fontsize=12)
    ax.set_ylabel("True Positive Rate", fontsize=12)
    ax.set_title(f"Scaffold Split — {cond_label}", fontweight="bold", fontsize=12)
    ax.legend(loc="lower right", fontsize=9, framealpha=.9)
    ax.grid(True, alpha=.3)
plt.suptitle("AUROC Curves — Scaffold Split | Anticancer | GCN/GAT/GT/MSMP",
             fontweight="bold", fontsize=14)
plt.tight_layout(); savefig("06_roc_scaffold_split.png")


In [ ]:
# ── Fig 07: Predicted vs True pIC50 — Random + Scaffold Aug ─────────────────
for ci, split_name, fname in [(1,"Random Aug","07_ic50_random_aug.png"),
                               (3,"Scaffold Aug","08_ic50_scaffold_aug.png")]:
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    for ax, mn, col in zip(axes, MODEL_NAMES, MODEL_COLORS):
        pl = PL[mn][ci]
        at = np.concatenate([fd[2] for fd in pl])
        ap = np.concatenate([fd[3] for fd in pl])
        r2   = r2_score(at, ap)
        rmse = float(np.sqrt(mean_squared_error(at, ap)))
        mae_ = float(mean_absolute_error(at, ap))
        ax.scatter(at, ap, alpha=.35, s=12, color=col)
        lims = [min(at.min(), ap.min())-.3, max(at.max(), ap.max())+.3]
        ax.plot(lims, lims, "k--", lw=1.5, alpha=.6)
        m_, b_ = np.polyfit(at, ap, 1)
        xl = np.linspace(lims[0], lims[1], 200)
        ax.plot(xl, m_*xl+b_, color=col, lw=2, alpha=.85)
        ax.set_xlim(lims); ax.set_ylim(lims)
        ax.set_xlabel("True pIC₅₀", fontsize=11)
        ax.set_ylabel("Predicted pIC₅₀", fontsize=11)
        ax.set_title(mn, fontweight="bold", fontsize=18, color=col, pad=10)
        ax.text(0.05, 0.95, f"R² = {r2:.4f}\nRMSE = {rmse:.4f}\nMAE = {mae_:.4f}",
                transform=ax.transAxes, fontsize=10, color="dimgray",
                verticalalignment="top",
                bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                          edgecolor="lightgray", alpha=0.85))
    plt.suptitle(f"Predicted vs True pIC₅₀ — {split_name} | Anticancer",
                 fontweight="bold", fontsize=13)
    plt.tight_layout(); savefig(fname)


In [ ]:
# ── Fig 09: Performance Heatmap ──────────────────────────────────────────────
r2_mat  = np.array([[np.mean([r2_score(fd[2], fd[3]) for fd in PL[mn][ci]])
                     for ci in range(4)] for mn in MODEL_NAMES])
acc_mat = np.array([[CLS_S[mn][ci]["accuracy"]["mean"]
                     for ci in range(4)] for mn in MODEL_NAMES])
auc_mat = np.array([[CLS_S[mn][ci]["auc"]["mean"]
                     for ci in range(4)] for mn in MODEL_NAMES])
mcc_mat = np.array([[CLS_S[mn][ci]["mcc"]["mean"]
                     for ci in range(4)] for mn in MODEL_NAMES])

fig, axes = plt.subplots(2, 2, figsize=(18, 10))
for ax, mat, title, cmap_ in zip(
        axes.flatten(),
        [r2_mat, acc_mat, auc_mat, mcc_mat],
        ["R² (Regression)", "Accuracy", "AUROC", "MCC"],
        ["YlGn", "Blues", "Purples", "Oranges"]):
    sns.heatmap(mat, annot=True, fmt=".4f", ax=ax, cmap=cmap_,
                xticklabels=COND_NAMES_LONG, yticklabels=MODEL_NAMES,
                linewidths=.5, linecolor="gray", vmin=.3, vmax=1.)
    bi, bj = np.unravel_index(mat.argmax(), mat.shape)
    ax.add_patch(plt.Rectangle((bj, bi), 1, 1,
                                fill=False, edgecolor="red", lw=3, zorder=5))
    ax.set_title(title, fontweight="bold", fontsize=12)
    ax.set_xlabel(
        f"★ Best: {MODEL_NAMES[bi]} | {COND_NAMES_LONG[bj]} = {mat[bi,bj]:.4f}",
        fontsize=9, color="red", fontweight="bold")
    ax.tick_params(axis="x", labelsize=8, rotation=20)
plt.suptitle("Performance Heatmaps — Best Cell Highlighted (red) | Anticancer",
             fontweight="bold", fontsize=13)
plt.tight_layout(); savefig("09_performance_heatmap.png")


In [ ]:
# ── Fig 10: Confusion Matrices — All Models (Scaffold Aug) ───────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, mn, col in zip(axes, MODEL_NAMES, MODEL_COLORS):
    cms = []
    for fd in PL[mn][3]:
        probs, labels = fd[0], fd[1]
        th = np.linspace(.1, .9, 161)
        ac = [accuracy_score(labels,(probs>=t).astype(int)) for t in th]
        cms.append(confusion_matrix(labels,
                                    (probs >= th[np.argmax(ac)]).astype(int)))
    mcm = np.mean(cms, 0)
    sns.heatmap(mcm, annot=True, fmt=".0f", ax=ax,
                cmap=sns.light_palette(col, as_cmap=True),
                xticklabels=["Pred Inactive","Pred Active"],
                yticklabels=["True Inactive","True Active"],
                linewidths=.5, linecolor="gray")
    ax.set_title(mn, fontweight="bold", fontsize=14, color=col)
plt.suptitle("Confusion Matrices — Mean over 5 Folds (Scaffold + Augmentation) | Anticancer",
             fontweight="bold", fontsize=13)
plt.tight_layout(); savefig("10_confusion_matrices.png")


In [ ]:
# ── Fig 11: Radar Chart — Scaffold Aug ───────────────────────────────────────
rkeys   = ["auc","accuracy","mcc","sensitivity","specificity",
           "f1","precision","balanced_accuracy"]
rlabels = ["AUROC","ACC","MCC","Sens","Spec","F1","Prec","BalACC"]
N_r     = len(rkeys)
angles  = np.linspace(0, 2*np.pi, N_r, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
for mn, col in zip(MODEL_NAMES, MODEL_COLORS):
    vals = [CLS_S[mn][3][k]["mean"] for k in rkeys]
    vals += [vals[0]]
    ax.plot(angles, vals, color=col, lw=2.2, marker="o",
            markersize=5, label=mn)
    ax.fill(angles, vals, color=col, alpha=.08)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(rlabels, fontsize=12, fontweight="bold")
ax.set_ylim(0, 1)
ax.set_yticks([.2,.4,.6,.8,1.])
ax.yaxis.grid(True, alpha=.4); ax.xaxis.grid(True, alpha=.4)
ax.set_title("Model Performance Radar — Scaffold Split + Augmentation | Anticancer",
             fontweight="bold", fontsize=12, pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=11)
plt.tight_layout(); savefig("11_radar_chart.png")


In [ ]:
# ── Fig 12: Training Loss Curves ─────────────────────────────────────────────
fig, axes = plt.subplots(len(MODEL_NAMES), 4,
                          figsize=(22, 4*len(MODEL_NAMES)))
for row, mn in enumerate(MODEL_NAMES):
    for col, ci in enumerate(range(4)):
        ax = axes[row, col]
        col_hex = MODEL_COLORS[row]
        all_losses = []
        for fold_h in HIST[mn][ci]:
            for seed, h in fold_h:
                all_losses.append([loss for _, loss, _ in h])
        if not all_losses: continue
        maxl = max(len(x) for x in all_losses)
        pad  = np.full((len(all_losses), maxl), np.nan)
        for i, x in enumerate(all_losses):
            for j, v in enumerate(x): pad[i, j] = v
        ml = np.nanmean(pad, 0); sl = np.nanstd(pad, 0)
        ax.plot(ml, color=col_hex, lw=2, label=mn)
        ax.fill_between(range(len(ml)), ml-sl, ml+sl,
                        color=col_hex, alpha=.2)
        ax.set_yscale("log")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
        ax.set_title(f"{mn} — {COND_NAMES_LONG[ci]}", fontweight="bold", fontsize=9)
        ax.legend(fontsize=8)
plt.suptitle("Training Loss Curves — All Models & Conditions | Anticancer",
             fontweight="bold", fontsize=13)
plt.tight_layout(); savefig("12_training_loss.png")


In [ ]:
# ── Fig 13: MSMP All 4 Conditions (Classification) ───────────────────────────
metrics_cls = ["accuracy","auc","mcc","sensitivity","specificity",
               "f1","precision","balanced_accuracy"]
fig, axes = plt.subplots(2, 4, figsize=(22, 10)); axes = axes.flatten()
for ax, m in zip(axes, metrics_cls):
    means = [CLS_S["MSMP"][c][m]["mean"] for c in range(4)]
    stds  = [CLS_S["MSMP"][c][m]["std"]  for c in range(4)]
    bars  = ax.bar(COND_LABELS, means, color=COND_COLORS, alpha=.88, edgecolor="white")
    ax.errorbar(range(4), means, yerr=stds, fmt="none", color="black", capsize=4, lw=1.5)
    ax.set_ylim(0, 1.18); ax.set_ylabel("Score"); ax.tick_params(axis="x", labelsize=8)
    ax.set_title(m.replace("_"," ").title(), fontweight="bold", fontsize=11)
    for b, v in zip(bars, means):
        ax.text(b.get_x()+b.get_width()/2, v+.01, f"{v:.3f}",
                ha="center", va="bottom", fontsize=8, fontweight="bold")
plt.suptitle("MSMP — Classification Metrics Across All 4 Conditions | Anticancer",
             fontweight="bold", fontsize=13, y=1.01)
plt.tight_layout(); savefig("13_msmp_classification_conditions.png")


In [ ]:
# ── Fig 14: MSMP All 4 Conditions (Regression) ───────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, m, title in zip(axes, ["r2","rmse","mae","mse"], ["R²","RMSE","MAE","MSE"]):
    means = [REG_S["MSMP"][c][m]["mean"] for c in range(4)]
    stds  = [REG_S["MSMP"][c][m]["std"]  for c in range(4)]
    bars  = ax.bar(COND_LABELS, means, color=COND_COLORS, alpha=.88, edgecolor="white")
    ax.errorbar(range(4), means, yerr=stds, fmt="none", color="black", capsize=4, lw=1.5)
    for b, v in zip(bars, means):
        ax.text(b.get_x()+b.get_width()/2, v+.005, f"{v:.3f}",
                ha="center", va="bottom", fontsize=9, fontweight="bold")
    ax.set_title(title, fontweight="bold", fontsize=12); ax.set_ylabel("Score")
    if m == "r2":
        ax.axhline(.70, color="red", ls="--", lw=1.6, label="R²=0.70 target")
        ax.legend(fontsize=9)
plt.suptitle("MSMP — Regression Metrics Across All 4 Conditions | Anticancer",
             fontweight="bold", fontsize=13)
plt.tight_layout(); savefig("14_msmp_regression_conditions.png")

print("\n✅ All figures saved to:", FIG_DIR)
print("\n" + "="*62)
print("  ANTICANCER GCN-AISM PIPELINE — COMPLETE")
print(f"  Best model   : {best_model_name} (Scaffold + Augmentation)")
print(f"  Threshold    : pIC50 >= {ACTIVITY_THRESHOLD}")
print(f"  Figures saved: {FIG_DIR}")
print(f"  Checkpoints  : {CKPT_DIR}")
print("="*62)


## 📊 Final Summary — Train & Test (2 decimal places)
> Run this cell anytime after training — no re-training needed.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 📊 FINAL SUMMARY — TRAIN & TEST METRICS (Rounded to 2 decimal places)
# Just run this cell after training — no re-training needed
# ════════════════════════════════════════════════════════════════════════════

import numpy as np
from sklearn.metrics import (accuracy_score, roc_auc_score, matthews_corrcoef,
    recall_score, confusion_matrix, balanced_accuracy_score, f1_score,
    precision_score, r2_score, mean_squared_error, mean_absolute_error)

def r2d(v): return round(float(v), 2)

cond_names = ["Rand-NoAug", "Rand-Aug", "Scaf-NoAug", "Scaf-Aug"]

# ── Compute TRAIN metrics from stored PL data ────────────────────────────────
# PL[mn][ci][fold] = (te_probs, te_labels, te_true_r, te_pred_r,
#                     tr_probs, tr_labels, tr_true_r, tr_pred_r)
def train_metrics_2d(mn, ci):
    cls_folds, reg_folds = [], []
    for fd in PL[mn][ci]:
        tp, tl, tr_r, pr_r = fd[4], fd[5], fd[6], fd[7]
        if len(set(tl.flatten())) < 2: continue
        th = np.linspace(0.1, 0.9, 161)
        ac = [accuracy_score(tl, (tp>=t).astype(int)) for t in th]
        preds = (tp >= th[np.argmax(ac)]).astype(int)
        try:
            tn, fp_, fn, tp2 = confusion_matrix(tl, preds).ravel()
            spec = tn/(tn+fp_) if (tn+fp_)>0 else 0.
        except:
            spec = 0.
        cls_folds.append({
            "ACC":    r2d(accuracy_score(tl, preds)),
            "AUC":    r2d(roc_auc_score(tl, tp)),
            "MCC":    r2d(matthews_corrcoef(tl, preds)),
            "Sens":   r2d(recall_score(tl, preds, zero_division=0)),
            "Spec":   r2d(spec),
            "F1":     r2d(f1_score(tl, preds, zero_division=0)),
            "Prec":   r2d(precision_score(tl, preds, zero_division=0)),
            "BalACC": r2d(balanced_accuracy_score(tl, preds))})
        reg_folds.append({
            "R2":   r2d(r2_score(tr_r, pr_r)),
            "RMSE": r2d(np.sqrt(mean_squared_error(tr_r, pr_r))),
            "MAE":  r2d(mean_absolute_error(tr_r, pr_r)),
            "MSE":  r2d(mean_squared_error(tr_r, pr_r))})
    def agg(lst):
        return {k: (r2d(np.mean([d[k] for d in lst])),
                    r2d(np.std( [d[k] for d in lst])))
                for k in lst[0]}
    return agg(cls_folds) if cls_folds else {}, agg(reg_folds) if reg_folds else {}

def test_metrics_2d(mn, ci):
    cs, rs = CLS_S[mn][ci], REG_S[mn][ci]
    cls_out = {k: (r2d(cs[k]["mean"]), r2d(cs[k]["std"]))
               for k in ["accuracy","auc","mcc","sensitivity",
                         "specificity","f1","precision","balanced_accuracy"]}
    reg_out = {k: (r2d(rs[k]["mean"]), r2d(rs[k]["std"]))
               for k in ["r2","rmse","mae","mse"]}
    return cls_out, reg_out

# ── Print helper ─────────────────────────────────────────────────────────────
def print_table(title, mode):
    print("\n" + "═"*105)
    print(f"  {title}")
    print("═"*105)
    print(f"  {'Model':<6} {'Condition':<13}  {'ACC':>7} {'AUC':>7} {'MCC':>7} "
          f"{'Sens':>7} {'Spec':>7} {'F1':>7} {'Prec':>7} {'BalACC':>7} "
          f"{'R²':>6} {'RMSE':>6} {'MAE':>6} {'MSE':>7}")
    print(f"  {'-'*99}")
    for mn in MODEL_NAMES:
        for ci, cn in enumerate(cond_names):
            if mode == "test":
                cc, rr = test_metrics_2d(mn, ci)
                ck = ["accuracy","auc","mcc","sensitivity","specificity","f1","precision","balanced_accuracy"]
                rk = ["r2","rmse","mae","mse"]
            else:
                cc, rr = train_metrics_2d(mn, ci)
                ck = ["ACC","AUC","MCC","Sens","Spec","F1","Prec","BalACC"]
                rk = ["R2","RMSE","MAE","MSE"]
            mk = "★" if ci in [1,3] else " "
            if not cc: continue
            cv = [f"{cc[k][0]:.2f}" for k in ck]
            rv = [f"{rr[k][0]:.2f}" for k in rk]
            print(f"  {mn:<6} {cn:<13}{mk}  " + "  ".join(f"{v:>5}" for v in cv)
                  + "  " + "  ".join(f"{v:>5}" for v in rv))
        print()

print_table("TEST SET — Classification + Regression  (mean over 5 folds, 2 decimal places)", "test")
print_table("TRAIN SET — Classification + Regression  (mean over 5 folds, 2 decimal places)", "train")

# ── Augmentation delta ────────────────────────────────────────────────────────
print("\n" + "═"*80)
print("  AUGMENTATION EFFECT  Δ = Aug − NoAug  (2 decimal places)  [TEST SET]")
print("═"*80)
for sname, no_ci, aug_ci in [("Random", 0, 1), ("Scaffold", 2, 3)]:
    print(f"\n  ── {sname} Split ──")
    print(f"  {'Model':<6}  {'ACC-Δ':>7} {'AUC-Δ':>7} {'MCC-Δ':>7} "
          f"{'F1-Δ':>7} {'BalACC-Δ':>9} {'R²-Δ':>7} {'RMSE-Δ':>8}")
    print(f"  {'-'*68}")
    for mn in MODEL_NAMES:
        cc_no, rr_no = test_metrics_2d(mn, no_ci)
        cc_au, rr_au = test_metrics_2d(mn, aug_ci)
        ck = ["accuracy","auc","mcc","f1","balanced_accuracy"]
        deltas = [r2d(cc_au[k][0] - cc_no[k][0]) for k in ck]
        r2d_  = r2d(rr_au["r2"][0]   - rr_no["r2"][0])
        rmsd  = r2d(rr_au["rmse"][0] - rr_no["rmse"][0])
        print(f"  {mn:<6}  " +
              "  ".join(f"{d:+.2f}" for d in deltas) +
              f"  {r2d_:+.2f}  {rmsd:+.2f}")

print("\n  ★ = Aug conditions | Cls metrics: +ve better | RMSE/MSE/MAE: -ve better")


## 📋 PPT Table Format — Train & Test Results
> Outputs the 4 tables matching the PPT format: **AUC | SN | SP | ACC | MCC | RMSE | R² | MAE** for both Training and Test datasets across all 4 conditions.\n
> Also exports a `ppt_table_results.csv` for easy copy-paste into the slides.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 📋 PPT TABLE FORMAT — Train & Test Results
# Matches the 4-table format: AUC | SN | SP | ACC | MCC | RMSE | R² | MAE
# Table 1: Random | No Aug   Table 2: Random | Aug
# Table 3: Scaffold | No Aug Table 4: Scaffold | Aug
# ════════════════════════════════════════════════════════════════════════════

import numpy as np, pandas as pd
from sklearn.metrics import (accuracy_score, roc_auc_score, matthews_corrcoef,
    recall_score, confusion_matrix, r2_score, mean_squared_error, mean_absolute_error)

def r4(v): return round(float(v), 4)

# ── Helper: compute train metrics from stored PL folds ───────────────────────
def get_train_metrics(mn, ci):
    auc_l, sn_l, sp_l, acc_l, mcc_l, rmse_l, r2_l, mae_l = [], [], [], [], [], [], [], []
    for fd in PL[mn][ci]:
        tp, tl, tr_r, pr_r = fd[4], fd[5], fd[6], fd[7]
        if len(set(np.array(tl).flatten())) < 2:
            continue
        tl_arr = np.array(tl).flatten()
        tp_arr = np.array(tp).flatten()
        tr_arr = np.array(tr_r).flatten()
        pr_arr = np.array(pr_r).flatten()
        th_vals = np.linspace(0.1, 0.9, 161)
        ac_vals = [accuracy_score(tl_arr, (tp_arr >= t).astype(int)) for t in th_vals]
        best_th = th_vals[np.argmax(ac_vals)]
        preds = (tp_arr >= best_th).astype(int)
        try:
            tn, fp_, fn, tp2 = confusion_matrix(tl_arr, preds).ravel()
            spec = tn / (tn + fp_) if (tn + fp_) > 0 else 0.
        except:
            spec = 0.
        auc_l.append(roc_auc_score(tl_arr, tp_arr))
        sn_l.append(recall_score(tl_arr, preds, zero_division=0))
        sp_l.append(spec)
        acc_l.append(accuracy_score(tl_arr, preds))
        mcc_l.append(matthews_corrcoef(tl_arr, preds))
        rmse_l.append(float(np.sqrt(mean_squared_error(tr_arr, pr_arr))))
        r2_l.append(r2_score(tr_arr, pr_arr))
        mae_l.append(float(mean_absolute_error(tr_arr, pr_arr)))
    if not auc_l:
        return {k: ("—", "—") for k in ["AUC","SN","SP","ACC","MCC","RMSE","R2","MAE"]}
    def m(lst): return r4(np.mean(lst))
    return {
        "AUC":  m(auc_l),  "SN":  m(sn_l),  "SP":  m(sp_l),
        "ACC":  m(acc_l),  "MCC": m(mcc_l),  "RMSE": m(rmse_l),
        "R2":   m(r2_l),   "MAE": m(mae_l),
    }

def get_test_metrics(mn, ci):
    cs = CLS_S[mn][ci]
    rs = REG_S[mn][ci]
    return {
        "AUC":  r4(cs["auc"]["mean"]),
        "SN":   r4(cs["sensitivity"]["mean"]),
        "SP":   r4(cs["specificity"]["mean"]),
        "ACC":  r4(cs["accuracy"]["mean"]),
        "MCC":  r4(cs["mcc"]["mean"]),
        "RMSE": r4(rs["rmse"]["mean"]),
        "R2":   r4(rs["r2"]["mean"]),
        "MAE":  r4(rs["mae"]["mean"]),
    }

COLS = ["AUC", "SN", "SP", "ACC", "MCC", "RMSE", "R2", "MAE"]
# Map model names to GNN labels for PPT
GNN_LABELS = {"GCN": "GNN-1", "GAT": "GNN-2", "GT": "GNN-3", "MSMP": "GNN-4"}

# Condition index map: 0=Rand-NoAug, 1=Rand-Aug, 2=Scaf-NoAug, 3=Scaf-Aug
TABLE_CONFIG = [
    (1, "Table 1", "Random Split | No Augmentation"),
    (2, "Table 2", "Random Split | With Augmentation"),
    (3, "Table 3", "Scaffold Split | No Augmentation"),
    (4, "Table 4", "Scaffold Split | With Augmentation"),
]
CI_MAP = {1: 0, 2: 1, 3: 2, 4: 3}

rows_all = []

for tnum, tname, tdesc in TABLE_CONFIG:
    ci = CI_MAP[tnum]
    print()
    print("=" * 100)
    print(f"  {tname}. Predictive performance of Graph based models — {tdesc}")
    print("=" * 100)

    # Header
    hdr = f"  {'Model':<8}  " + " | ".join([
        f"{'── Training Dataset ──':^56}",
        f"{'── Independent Test Dataset ──':^56}"
    ])
    col_hdr = f"  {'':8}  " + "  ".join([f"{c:>6}" for c in COLS]) + "    " + "  ".join([f"{c:>6}" for c in COLS])
    print(col_hdr)
    print("  " + "-" * 96)

    table_rows = []
    for mn in MODEL_NAMES:
        label = GNN_LABELS[mn]
        tr_m = get_train_metrics(mn, ci)
        te_m = get_test_metrics(mn, ci)
        tr_vals = "  ".join([f"{tr_m[c]:>6}" if isinstance(tr_m[c], float) else f"{'—':>6}" for c in COLS])
        te_vals = "  ".join([f"{te_m[c]:>6}" if isinstance(te_m[c], float) else f"{'—':>6}" for c in COLS])
        row_str = f"  {label:<8}  {tr_vals}    {te_vals}"
        print(row_str)
        table_rows.append({
            "Table": tname,
            "Condition": tdesc,
            "Model": label,
            **{f"Train_{c}": (tr_m[c] if isinstance(tr_m[c], float) else None) for c in COLS},
            **{f"Test_{c}":  (te_m[c] if isinstance(te_m[c], float) else None) for c in COLS},
        })
    rows_all.extend(table_rows)

print()
print("✅ PPT table data computed. Saving to CSV…")

df_ppt = pd.DataFrame(rows_all)
csv_path = "./ppt_table_results.csv"
df_ppt.to_csv(csv_path, index=False)
print(f"   Saved → {csv_path}")
print()
print("Column legend: SN = Sensitivity, SP = Specificity, R2 = R²")
print("GNN-1=GCN | GNN-2=GAT | GNN-3=GT | GNN-4=MSMP")


In [ ]:
import json

deployment_config = {
    "node_dim": NODE_DIM,
    "desc_dim": DESC_DIM,
    "y_reg_mean": float(y_reg_mean),
    "y_reg_std": float(y_reg_std),
    "epochs": EPOCHS,
    "patience": PATIENCE,
}

with open("/kaggle/working/deployment_config.json", "w") as f:
    json.dump(deployment_config, f, indent=2)

print("✅ deployment_config.json saved")

In [ ]:
import subprocess

subprocess.run(
    "pip freeze > /kaggle/working/requirements.txt",
    shell=True
)

print("✅ requirements.txt saved")